In [15]:
from pathlib import Path
import os
import sys
import subprocess
import importlib.util

PROJECT_ROOT = Path.cwd().resolve()

if not (PROJECT_ROOT / "quran_asr").exists() or not (PROJECT_ROOT / "configs").exists():
    raise RuntimeError(
        f"Notebook harus dijalankan dari root project.\n"
        f"Posisi sekarang: {PROJECT_ROOT}\n\n"
        "Harusnya dari:\n"
        "/home/jrilym/Projects/AI/model-asr-quran"
    )

CONFIG_PATH = PROJECT_ROOT / "configs" / "local_3050.yaml"

if not CONFIG_PATH.exists():
    raise FileNotFoundError(f"Config belum ada: {CONFIG_PATH}")

CACHE_DIR = PROJECT_ROOT / ".cache" / "hf"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

os.environ["HF_HOME"] = str(CACHE_DIR)
os.environ["HF_HUB_CACHE"] = str(CACHE_DIR / "hub")
os.environ["TRANSFORMERS_CACHE"] = str(CACHE_DIR / "transformers")

print("PROJECT_ROOT:", PROJECT_ROOT)
print("CONFIG_PATH:", CONFIG_PATH)
print("HF_HOME:", os.environ["HF_HOME"])
print("Python:", sys.executable)

PROJECT_ROOT: /home/jrilym/Projects/AI/model-asr-quran
CONFIG_PATH: /home/jrilym/Projects/AI/model-asr-quran/configs/local_3050.yaml
HF_HOME: /home/jrilym/Projects/AI/model-asr-quran/.cache/hf
Python: /home/jrilym/Projects/AI/model-asr-quran/.venv/bin/python


In [17]:
import quran_asr
import yaml

print("quran_asr:", quran_asr.__file__)
print("Dependency download OK")

quran_asr: /home/jrilym/Projects/AI/model-asr-quran/quran_asr/__init__.py
Dependency download OK


In [18]:
def run_cmd(args):
    print("Running:")
    print(" ".join(str(arg) for arg in args))
    subprocess.run(args, check=True, cwd=PROJECT_ROOT)

In [19]:
RAW_DIR = PROJECT_ROOT / "data" / "raw"
AUDIO_DIR = RAW_DIR / "audio"
TEXT_DIR = RAW_DIR / "text"
BACKUP_DIR = PROJECT_ROOT / "backups"
AUDIO_BACKUP = BACKUP_DIR / "audio.tar"

RAW_DIR.mkdir(parents=True, exist_ok=True)
AUDIO_DIR.mkdir(parents=True, exist_ok=True)
TEXT_DIR.mkdir(parents=True, exist_ok=True)
BACKUP_DIR.mkdir(parents=True, exist_ok=True)

print("RAW_DIR:", RAW_DIR)
print("AUDIO_DIR:", AUDIO_DIR)
print("TEXT_DIR:", TEXT_DIR)
print("BACKUP_DIR:", BACKUP_DIR)

RAW_DIR: /home/jrilym/Projects/AI/model-asr-quran/data/raw
AUDIO_DIR: /home/jrilym/Projects/AI/model-asr-quran/data/raw/audio
TEXT_DIR: /home/jrilym/Projects/AI/model-asr-quran/data/raw/text
BACKUP_DIR: /home/jrilym/Projects/AI/model-asr-quran/backups


In [20]:
has_audio = AUDIO_DIR.is_dir() and any(AUDIO_DIR.iterdir())
has_backup = AUDIO_BACKUP.exists()

def backup_audio():
    print("Backing up audio locally...")
    subprocess.run(
        ["tar", "cf", str(AUDIO_BACKUP), "-C", str(RAW_DIR), "audio"],
        check=True,
        cwd=PROJECT_ROOT,
    )

if has_audio and has_backup:
    print("Audio and local backup are already present.")

elif has_audio:
    print("Audio already exists. Creating local backup...")
    backup_audio()

elif has_backup:
    print("Restoring audio from local backup...")
    subprocess.run(
        ["tar", "xf", str(AUDIO_BACKUP), "-C", str(RAW_DIR)],
        check=True,
        cwd=PROJECT_ROOT,
    )

else:
    print("First run: downloading MP3 audio and Quran text...")
    run_cmd([
        sys.executable,
        str(PROJECT_ROOT / "scripts" / "download.py"),
        "--config",
        str(CONFIG_PATH),
        "--no-convert",
        "--max-workers",
        "8",
        "--qps",
        "10",
    ])

    backup_audio()

Audio and local backup are already present.


In [21]:
text_file = PROJECT_ROOT / "data" / "raw" / "text" / "quran_uthmani.json"
audio_files = list(AUDIO_DIR.rglob("*.*"))

print("Quran text:", "OK" if text_file.exists() else "MISSING", "-", text_file)
print("Audio dir:", "OK" if AUDIO_DIR.exists() else "MISSING", "-", AUDIO_DIR)
print("Jumlah file audio:", len(audio_files))

print("\nContoh file audio:")
for file in audio_files[:10]:
    print("-", file.relative_to(PROJECT_ROOT))

Quran text: OK - /home/jrilym/Projects/AI/model-asr-quran/data/raw/text/quran_uthmani.json
Audio dir: OK - /home/jrilym/Projects/AI/model-asr-quran/data/raw/audio
Jumlah file audio: 24946

Contoh file audio:
- data/raw/audio/Husary_128kbps_Mujawwad/001003.mp3
- data/raw/audio/Husary_128kbps_Mujawwad/001003.md5
- data/raw/audio/Husary_128kbps_Mujawwad/001002.mp3
- data/raw/audio/Husary_128kbps_Mujawwad/001002.md5
- data/raw/audio/Husary_128kbps_Mujawwad/001001.mp3
- data/raw/audio/Husary_128kbps_Mujawwad/001001.md5
- data/raw/audio/Husary_128kbps_Mujawwad/001004.mp3
- data/raw/audio/Husary_128kbps_Mujawwad/001004.md5
- data/raw/audio/Husary_128kbps_Mujawwad/001005.mp3
- data/raw/audio/Husary_128kbps_Mujawwad/001005.md5


In [22]:
subprocess.run(["du", "-sh", str(PROJECT_ROOT / "data" / "raw")], check=False)
subprocess.run(["du", "-sh", str(AUDIO_DIR)], check=False)
subprocess.run(["du", "-sh", str(BACKUP_DIR)], check=False)

4.9G	/home/jrilym/Projects/AI/model-asr-quran/data/raw
4.9G	/home/jrilym/Projects/AI/model-asr-quran/data/raw/audio
4.8G	/home/jrilym/Projects/AI/model-asr-quran/backups


CompletedProcess(args=['du', '-sh', '/home/jrilym/Projects/AI/model-asr-quran/backups'], returncode=0)